In [ ]:
# Installs pySDC where it is missing, e.g. on Colab
try:
    import pySDC
except ImportError:
    %pip install -q git+https://github.com/Parallel-in-Time/pySDC.git

# Part G: pySDC from Firedrake forms

In [Part D](D_pySDC_with_Firedrake), the heat equation comes with a ready-made problem class. Here we write our own
problem without writing a class at all: we write down the weak form of the equation, as we would for Firedrake
anyway, and the generic problem class
[`FormFiredrake`](https://github.com/Parallel-in-Time/pySDC/blob/master/pySDC/implementations/problem_classes/FormFiredrake.py)
builds everything pySDC needs from it. Part D's heat equation is itself built this way.

## The equation and its weak form

We solve a reaction-diffusion equation of Allen-Cahn type on the unit square,

$$
u_t = \nu \Delta u + u - u^3 + g,
$$

with Dirichlet boundary values. The forcing $g$ and the boundary values come from a solution $u_e(x, y, t)$ we pick,
so that we can measure the error: this is the *method of manufactured solutions*.

Finite elements turn the equation into a system of ordinary differential equations, $M u' = R(u)$, for the
coefficients $u$ of the finite element function. The *mass matrix* $M$ and the right-hand side $R$ come from
multiplying the equation by a test function $v$ and integrating over the domain, integrating the Laplacian by parts:

$$
\underbrace{\int u_t\, v\, dx}_{m(u_t, v)}
= \underbrace{-\nu \int \nabla u \cdot \nabla v\, dx}_{\text{diffusion}}
+ \underbrace{\int (u - u^3)\, v\, dx}_{\text{reaction}} + \int g\, v\, dx.
$$

These *forms* are all `FormFiredrake` needs. We give each as a Python function of the solution `u` and the test
function `v`, written in [UFL](https://www.firedrakeproject.org/variational-problems.html), the form language
Firedrake uses. Time enters through a Firedrake `Constant` `t`: pySDC sets it to the right time before every
evaluation and solve, so forms, boundary values and the exact solution may all depend on it.

The forcing is the one part we do not write by hand. `manufactured_forcing` inserts $u_e$ into the forms, and UFL
differentiates it in time, so $g$ is exact for whatever $u_e$ we choose. Ours is quadratic in space, so the finite
elements represent it exactly and all of the error we see comes from the time stepping.

In [ ]:
from pathlib import Path

import firedrake as fd

from pySDC.helpers.ufl_forms import manufactured_forcing, total_rhs

NU = 0.1


def get_forms(imex, mesh):
    """
    The problem as forms, for `FormFiredrake` or `FormFiredrakeIMEX`.

    Args:
        imex (bool): Whether to put the reaction into the explicit part
        mesh (firedrake.MeshGeometry): The mesh

    Returns:
        dict: The problem parameters
    """
    V = fd.FunctionSpace(mesh, 'CG', 2)
    x, y = fd.SpatialCoordinate(mesh)
    t = fd.Constant(0.0)

    u_exact = fd.cos(t) * (1 + x**2 + 2 * y**2) / 4

    def mass(u, v):
        return u * v * fd.dx

    def diffusion(u, v):
        return -NU * fd.inner(fd.grad(u), fd.grad(v)) * fd.dx

    def reaction(u, v):
        return (u - u**3) * v * fd.dx

    forcing = manufactured_forcing(mass, total_rhs(diffusion, reaction), u_exact, t, fd.TestFunction(V))

    params = {
        'V': V,
        'mass': mass,
        't': t,
        'u_exact': u_exact,
        # a tuple: pySDC would read a list as one entry per level
        'bcs': (fd.DirichletBC(V, u_exact, 'on_boundary'),),
    }
    if imex:
        params['rhs'] = lambda u, v: diffusion(u, v) + forcing
        params['rhs_exp'] = reaction
    else:
        params['rhs'] = lambda u, v: diffusion(u, v) + reaction(u, v) + forcing
    return params

## Implicit or IMEX

`FormFiredrake` treats the whole right-hand side implicitly, so every SDC node needs a nonlinear solve. Firedrake
does it with Newton's method, and the Jacobian comes from differentiating the forms symbolically. `FormFiredrakeIMEX`
takes a second, explicit right-hand side: here the reaction, so the implicit systems become linear. The problem class
notices that from the forms: it then solves each system with one linear solve instead of Newton's method, and since
the operator does not depend on time either, it factorizes it once per prefactor and keeps the factorization. Either
way, the problem class never inverts the mass matrix. It hands pySDC $M u$ and $R(u)$ instead, which is what the
*mass-matrix sweepers* `generic_implicit_mass` and `imex_1st_order_mass` work with.

The boundary values enter only in the solves. As a result, the rows of the SDC residual that belong to boundary
nodes carry no equation, and the problem class leaves them out of the residual it reports.

The residual is a load vector like $R(u)$, and its size is measured with the norm that belongs to load vectors: the
$L^2$ norm of the function it represents, $\sqrt{r^T M^{-1} r}$. It does not shrink as the mesh is refined, so a
residual tolerance means the same on every mesh. Here, `restol = 1e-8` is where the iterations reach the accuracy
of the collocation problem; a smaller one only adds iterations.

In [ ]:
def setup(imex, levels=1):
    """
    Helper routine to set up parameters

    Args:
        imex (bool): Whether to run IMEX SDC instead of fully implicit SDC
        levels (int): Number of levels, 1 for SDC or 2 for MLSDC

    Returns:
        description and controller_params parameter dictionaries
    """
    from pySDC.implementations.problem_classes.FormFiredrake import FormFiredrake, FormFiredrakeIMEX
    from pySDC.implementations.sweeper_classes.generic_implicit_mass import generic_implicit_mass
    from pySDC.implementations.sweeper_classes.imex_1st_order_mass import imex_1st_order_mass
    from pySDC.implementations.transfer_classes.BaseTransfer_mass import base_transfer_mass
    from pySDC.implementations.transfer_classes.TransferFiredrakeMesh import MeshToMeshFiredrakeHierarchy
    from pySDC.implementations.hooks.log_errors import LogGlobalErrorPostRun
    from pySDC.implementations.hooks.log_work import LogWork

    # an 8x8 mesh made by refining a 4x4 one, so that the two can be levels
    meshes = fd.MeshHierarchy(fd.UnitSquareMesh(4, 4), 1)[::-1][:levels]
    forms = [get_forms(imex, mesh) for mesh in meshes]

    description = {
        'problem_class': FormFiredrakeIMEX if imex else FormFiredrake,
        # one entry per level, finest first: each level has its own mesh, and so its own spaces and boundary values
        'problem_params': {key: [level[key] for level in forms] for key in forms[0]},
        'sweeper_class': imex_1st_order_mass if imex else generic_implicit_mass,
        'sweeper_params': {'quad_type': 'RADAU-RIGHT', 'num_nodes': 3, 'QI': 'LU'},
        'level_params': {'restol': 1e-8, 'dt': 0.1},
        'step_params': {'maxiter': 20},
        'base_transfer_class': base_transfer_mass,
        'space_transfer_class': MeshToMeshFiredrakeHierarchy,
    }
    controller_params = {'logger_level': 30, 'hook_class': [LogGlobalErrorPostRun, LogWork]}
    return description, controller_params

## Several levels

For multilevel SDC (MLSDC) and PFASST, each level needs its own problem on its own mesh. We take the meshes from a
Firedrake `MeshHierarchy`, so that they are nested, and give the problem parameters as lists, one entry per level,
finest first. Two transfer classes move data between the levels:

- `MeshToMeshFiredrakeHierarchy` restricts solutions by $L^2$ projection, restricts load vectors such as $M u_0$ with
  the transpose of the prolongation, and prolongs corrections;
- `base_transfer_mass` puts these together into the FAS correction for the mass formulation.

On the coarse level, the solves impose the boundary values of the restricted fine solution rather than the boundary
data, so that the coarse correction vanishes on the boundary. Without this, MLSDC would not converge to the solution
of SDC.

## Running it

The run prints the error at the end, the SDC iterations and the work of the problem class: how many solvers it set
up (one per distinct diagonal entry of the preconditioner), how many systems it solved, and how many linear solves
those took in total, including Newton's.

In [ ]:
def run(imex, levels=1):
    """
    Run the example with fully implicit or IMEX SDC, or MLSDC

    Args:
        imex (bool): Whether to run IMEX SDC instead of fully implicit SDC
        levels (int): Number of levels, 1 for SDC or 2 for MLSDC

    Returns:
        dict: Error, iterations and work of the run
    """
    import numpy as np
    from pySDC.implementations.controller_classes.controller_nonMPI import controller_nonMPI
    from pySDC.helpers.stats_helper import get_sorted

    description, controller_params = setup(imex, levels)
    controller = controller_nonMPI(num_procs=1, controller_params=controller_params, description=description)

    P = controller.MS[0].levels[0].prob
    uend, stats = controller.run(u0=P.u_exact(0.0), t0=0.0, Tend=1.0)

    result = {'error': get_sorted(stats, type='e_global_post_run')[0][1]}
    result['niter'] = int(np.sum([me[1] for me in get_sorted(stats, type='niter')]))
    for key in ['work_solver_setup', 'work_solver', 'work_ksp']:
        # the work on the finest level
        result[key] = int(np.sum([me[1] for me in get_sorted(stats, type=key, level=0)]))

    variant = ('IMEX' if imex else 'implicit') + (' SDC' if levels == 1 else ' MLSDC')
    summary = (
        f'{variant:14s}: error {result["error"]:.2e}, {result["niter"]} iterations, '
        f'{result["work_solver_setup"]} solver setups, {result["work_solver"]} solves, '
        f'{result["work_ksp"]} linear solves'
    )
    print(summary)
    Path('data').mkdir(parents=True, exist_ok=True)
    with open('data/step_7_G_out.txt', 'a') as file:
        file.write(summary + '\n')
    return result


def main():
    """Run the variants and check that we got the same as last time."""
    implicit = run(imex=False)
    imex = run(imex=True)
    multilevel = run(imex=False, levels=2)

    for result in [implicit, imex, multilevel]:
        assert result['error'] < 1e-8, result
        assert result['work_solver_setup'] == 3, result
        assert result['work_solver'] == 3 * result['niter'], result

    # the reaction is mild enough that moving it to the explicit side costs no extra iterations here
    assert implicit['niter'] == 61 and imex['niter'] == 59, (implicit, imex)
    # the IMEX systems are linear: one linear solve each, while Newton needs a few
    assert imex['work_ksp'] == imex['work_solver'], imex
    assert implicit['work_ksp'] > implicit['work_solver'], implicit
    # MLSDC reaches the same accuracy, in about as many iterations as SDC, see below
    assert multilevel['niter'] == 66, multilevel
    return implicit, imex, multilevel

In [ ]:
if __name__ == "__main__":
    main()

All variants reach the same accuracy. IMEX SDC solves one linear system per node and iteration, where fully implicit
SDC runs Newton's method. Which one is cheaper overall depends on how stiff the explicit part is: a stiffer reaction
would cost IMEX SDC iterations, or a smaller step size.

MLSDC does not pay off for this problem: SDC already converges in about six iterations per step, and MLSDC needs as
many, plus the sweeps on the coarse level. A coarse level helps when SDC needs many iterations, for instance with
stronger diffusion or larger steps, and it is what lets PFASST compute several steps in parallel.

Firedrake does not run in the browser, nor in the environment this website is built in. Our CI runs this part in a
Firedrake container and checks its errors and work counts. These are its results, in the run that built this page:

:::{literalinclude} /../../data_firedrake/step_7_G_out.txt
:language: text
:::